# HMS - Harmful Brain Activity Classification
## PyTorch Pipeline: nnAudio (Trainable STFT) + EfficientNet + GRU

This notebook implements a full training and inference pipeline that:
1. Loads **raw EEG** signals (no pre-computed spectrograms)
2. Computes spectrograms **on-the-fly** using nnAudio with a trainable STFT
3. Extracts features using a pretrained **EfficientNetV2-B2** backbone
4. Models temporal evolution with a **Bidirectional GRU**
5. Classifies into 6 harmful brain activity categories

### Pipeline
```
Raw EEG [batch, 16, 10000]
    ↓
nnAudio STFT (trainable)
    ↓
Log + Normalize
    ↓
Mono → 3 channel
    ↓
EfficientNet backbone (remove classification head)
    ↓
Pool over frequency, keep time axis
    ↓
GRU reads across time
    ↓
Softmax → 6 classes
```

### Reference
This notebook is the PyTorch counterpart of the Keras starter notebook.
Both can be ensembled at the prediction level (numpy arrays).

# 🛠 | Install Libraries

Run this cell once to install required packages.

In [ ]:
# All dependencies are installed via requirements.txt:
#   pip install -r requirements.txt
#
# If running inside the notebook without requirements.txt:
# !pip install -r requirements.txt

# 📚 | Import Libraries

In [ ]:
import os
import math
import random
import gc
from pathlib import Path
from glob import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

import timm
from nnAudio.features import STFT

from sklearn.model_selection import StratifiedGroupKFold
import joblib

print(f"PyTorch: {torch.__version__}")
print(f"timm: {timm.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ⚙️ | Configuration

Mirrors the Keras notebook's CFG class with additions for STFT, GRU,
and differential learning rates.

In [ ]:
class CFG:
    # --- General ---
    verbose = 1
    seed = 42
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # --- Data ---
    num_classes = 6
    class_names = ["Seizure", "LPD", "GPD", "LRDA", "GRDA", "Other"]
    label2name = dict(enumerate(class_names))
    name2label = {v: k for k, v in label2name.items()}

    # --- EEG ---
    eeg_sample_rate = 200       # Hz
    eeg_duration = 50           # seconds
    eeg_samples = eeg_sample_rate * eeg_duration  # 10000
    num_bipolar_channels = 16   # 4 chains x 4 pairs

    # --- nnAudio STFT ---
    n_fft = 128
    hop_length = 128
    trainable_stft = True

    # --- Model ---
    backbone = "tf_efficientnetv2_s"
    gru_hidden = 128
    gru_layers = 2
    dropout = 0.35

    # --- Stage 1: Train on ALL data ---
    epochs_stage1 = 20
    batch_size = 16
    lr_backbone = 5e-5
    lr_stft = 5e-4
    lr_head = 3e-4
    weight_decay = 1e-4
    max_grad_norm = 1.0

    # --- Stage 2: Fine-tune on HIGH-QUALITY data (≥min_votes) ---
    epochs_stage2 = 10
    lr_stage2_factor = 0.3      # multiply stage1 LRs by this
    min_votes_stage2 = 10       # minimum total expert votes

    # --- Derived (set by code) ---
    epochs = epochs_stage1      # used by scheduler/progress bars

    # --- General Training ---
    lr_mode = "cos"
    use_amp = False
    num_workers = 4
    n_folds = 5
    run_folds = [0, 1, 2, 3, 4]  # which folds to train (all 5 for full CV)

    # --- Labels ---
    use_soft_labels = True
    label_smoothing = 0.05

    # --- SpecAugment ---
    spec_freq_mask = 10
    spec_time_mask = 10
    spec_num_masks = 2

print(f"Device: {CFG.device}")
print(f"Stage 1: {CFG.epochs_stage1} epochs on ALL data")
print(f"Stage 2: {CFG.epochs_stage2} epochs on high-quality (≥{CFG.min_votes_stage2} votes)")
print(f"Stage 2 LR factor: {CFG.lr_stage2_factor}x")
print(f"Folds to run: {CFG.run_folds}")
print(f"LRs: stft={CFG.lr_stft}, backbone={CFG.lr_backbone}, head={CFG.lr_head}")
print(f"STFT: n_fft={CFG.n_fft}, hop_length={CFG.hop_length}")

# ♻️ | Reproducibility

Sets value for random seed to produce similar result in each run.

In [ ]:
def set_seed(seed=CFG.seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

# 📁 | Dataset Paths

**Important**: Adjust `BASE_PATH` to match your data location.
The Keras notebook uses `project_root.parent.parent / "data" / "data"`.

In [ ]:
project_root = Path.cwd()

# =============================================
# ADJUST THIS PATH TO MATCH YOUR DATA LOCATION
# This matches the Keras notebook's path structure
# =============================================
BASE_PATH = project_root.parent.parent.parent / "data" / "data"

# Output directories
MODELS_DIR = project_root / "models"
RESULTS_DIR = project_root / "results"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Verify
print("=" * 60)
print(f"Project root: {project_root}")
print(f"Data path:    {BASE_PATH}")
print(f"Data exists:  {BASE_PATH.exists()}")

if BASE_PATH.exists():
    for name in ["train.csv", "test.csv", "train_eegs", "test_eegs"]:
        path = BASE_PATH / name
        status = "✓" if path.exists() else "✗"
        print(f"  {status} {name}")
print("=" * 60)

# Pre-processed bipolar EEG files (.npy)
PROCESSED_DIR = BASE_PATH / "processed" / "bipolar_eegs"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print(f"Processed EEGs: {PROCESSED_DIR}")


# 📖 | Metadata

We only need the EEG paths (not spectrogram paths) since nnAudio
generates spectrograms on-the-fly from raw EEG.

In [ ]:
# --- Train ---
df = pd.read_csv(BASE_PATH / "train.csv")
df["eeg_path"] = df["eeg_id"].apply(
    lambda x: str(BASE_PATH / "train_eegs" / f"{x}.parquet")
)
df["class_name"] = df["expert_consensus"].copy()
df["class_label"] = df["expert_consensus"].map(CFG.name2label)

# Soft labels + total votes from expert vote counts
vote_cols = ["seizure_vote", "lpd_vote", "gpd_vote",
             "lrda_vote", "grda_vote", "other_vote"]
if all(c in df.columns for c in vote_cols):
    votes = df[vote_cols].values.astype(np.float32)
    vote_sums = votes.sum(axis=1, keepdims=True)
    vote_sums_safe = np.clip(vote_sums, 1.0, None)
    df["soft_labels"] = (votes / vote_sums_safe).tolist()
    df["total_votes"] = vote_sums.flatten().astype(int)
    print(f"✓ Soft labels created from {vote_cols}")
    print(f"  Total votes distribution:")
    print(f"    min={df['total_votes'].min()}, median={df['total_votes'].median():.0f}, "
          f"max={df['total_votes'].max()}")
    hq_count = (df["total_votes"] >= CFG.min_votes_stage2).sum()
    print(f"  High-quality samples (≥{CFG.min_votes_stage2} votes): {hq_count} "
          f"({hq_count/len(df)*100:.1f}%)")
else:
    df["soft_labels"] = None
    df["total_votes"] = 1
    print("⚠ Vote columns not found — will use one-hot labels")

print(f"\nTrain CSV rows: {len(df)}")
print(f"Unique EEG files: {df['eeg_id'].nunique()}")
print(f"Unique patients: {df['patient_id'].nunique()}")
display(df.head(2))

# --- Test ---
test_df = pd.read_csv(BASE_PATH / "test.csv")
test_df["eeg_path"] = test_df["eeg_id"].apply(
    lambda x: str(BASE_PATH / "test_eegs" / f"{x}.parquet")
)

print(f"\nTest CSV rows: {len(test_df)}")
display(test_df.head(2))

# 🧠 | Bipolar Banana Montage

The standard 10-20 system has 19 electrodes. We compute the **double banana bipolar montage**,
which subtracts adjacent electrodes to produce 16 bipolar channels in 4 chains:

| Chain | Pairs | Region |
|-------|-------|--------|
| LL (Left Lateral) | Fp1-F7, F7-T3, T3-T5, T5-O1 | Left temporal |
| RL (Right Lateral) | Fp2-F8, F8-T4, T4-T6, T6-O2 | Right temporal |
| LP (Left Parasagittal) | Fp1-F3, F3-C3, C3-P3, P3-O1 | Left central |
| RP (Right Parasagittal) | Fp2-F4, F4-C4, C4-P4, P4-O2 | Right central |

In [ ]:
BIPOLAR_MONTAGE = {
    "LL": [("Fp1", "F7"), ("F7", "T3"), ("T3", "T5"), ("T5", "O1")],
    "RL": [("Fp2", "F8"), ("F8", "T4"), ("T4", "T6"), ("T6", "O2")],
    "LP": [("Fp1", "F3"), ("F3", "C3"), ("C3", "P3"), ("P3", "O1")],
    "RP": [("Fp2", "F4"), ("F4", "C4"), ("C4", "P4"), ("P4", "O2")],
}

BIPOLAR_PAIRS = []
CHAIN_ORDER = ["LL", "RL", "LP", "RP"]
for chain_name in CHAIN_ORDER:
    BIPOLAR_PAIRS.extend(BIPOLAR_MONTAGE[chain_name])

print(f"Total bipolar channels: {len(BIPOLAR_PAIRS)}")
for i, (a, b) in enumerate(BIPOLAR_PAIRS):
    chain = CHAIN_ORDER[i // 4]
    print(f"  Ch {i:2d} ({chain}): {a} - {b}")

# Verify electrode names exist in an actual EEG file
sample_eeg = pd.read_parquet(df.iloc[0]["eeg_path"])
eeg_columns = sample_eeg.columns.tolist()
print(f"\nEEG file columns: {eeg_columns}")
needed = set()
for a, b in BIPOLAR_PAIRS:
    needed.add(a)
    needed.add(b)
missing = needed - set(eeg_columns)
if missing:
    print(f"⚠ WARNING: Missing electrodes: {missing}")
    print("You may need to adjust electrode names in BIPOLAR_MONTAGE")
else:
    print("✓ All required electrodes found in EEG data")

# 🔄 | Pre-process EEG → Bipolar .npz

One-time conversion: read each raw EEG parquet, compute bipolar montage,
normalize, and save as a lightweight `.npz` file.

Each file contains:
- `eeg`: normalized bipolar signal `[16, 10000]`
- `stats`: raw channel statistics `[32]` (16 means + 16 stds)

**Why save stats?** Normalization destroys amplitude information.
A seizure channel has high std, suppression has low std.
We preserve these as side features for the classification head.

In [ ]:
def preprocess_eeg(row_idx, dataframe, split="train"):
    """Convert one EEG parquet → pre-processed bipolar .npz file.
    Saves both normalized EEG [16, 10000] and raw channel stats [32].
    """
    row = dataframe.iloc[row_idx]
    eeg_id = row["eeg_id"]
    output_path = PROCESSED_DIR / f"{eeg_id}.npz"

    if output_path.exists():
        return  # skip if already done

    # Load raw EEG
    eeg_df = pd.read_parquet(row["eeg_path"])

    # Extract window
    offset = int(row.get("eeg_label_offset_seconds", 0))
    start = offset * CFG.eeg_sample_rate
    end = start + CFG.eeg_samples
    window = eeg_df.iloc[start:end]

    # Pad if needed
    if len(window) < CFG.eeg_samples:
        pad = pd.DataFrame(
            np.zeros((CFG.eeg_samples - len(window), len(window.columns))),
            columns=window.columns,
        )
        window = pd.concat([window, pad], ignore_index=True)

    # Compute bipolar montage
    cols = window.columns.tolist()
    bipolar = []
    for (a, b) in BIPOLAR_PAIRS:
        if a in cols and b in cols:
            sig = window[a].values - window[b].values
        else:
            sig = np.zeros(CFG.eeg_samples, dtype=np.float32)
        bipolar.append(sig)
    bipolar = np.stack(bipolar, axis=0).astype(np.float32)  # [16, 10000]

    # Clean
    bipolar = np.nan_to_num(bipolar, nan=0.0)
    bipolar = np.clip(bipolar, -1024, 1024)

    # Compute stats BEFORE normalizing (this is what normalization destroys)
    chan_mean = bipolar.mean(axis=1).astype(np.float32)  # [16]
    chan_std = bipolar.std(axis=1).astype(np.float32)    # [16]
    stats = np.concatenate([chan_mean, chan_std])          # [32]

    # Normalize per channel
    bipolar = (bipolar - chan_mean[:, None]) / (chan_std[:, None] + 1e-6)

    # Save both: normalized EEG + raw stats
    np.savez_compressed(str(output_path), eeg=bipolar, stats=stats)


# --- Process training EEGs ---
unique_train = df.drop_duplicates(subset="eeg_id").reset_index(drop=True)
print(f"Processing {len(unique_train)} unique training EEG files...")
_ = joblib.Parallel(n_jobs=-1, backend="loky")(
    joblib.delayed(preprocess_eeg)(i, unique_train, "train")
    for i in tqdm(range(len(unique_train)))
)

# --- Process test EEGs ---
print(f"Processing {len(test_df)} test EEG files...")
_ = joblib.Parallel(n_jobs=-1, backend="loky")(
    joblib.delayed(preprocess_eeg)(i, test_df, "test")
    for i in tqdm(range(len(test_df)))
)

sample_file = list(PROCESSED_DIR.glob('*.npz'))[0]
sample_data = np.load(str(sample_file))
print(f"\n✓ Pre-processing complete. Files saved to: {PROCESSED_DIR}")
print(f"  Sample file size: {sample_file.stat().st_size / 1024:.0f} KB")
print(f"  EEG shape: {sample_data['eeg'].shape}")
print(f"  Stats shape: {sample_data['stats'].shape}  (16 means + 16 stds)")

# 🍚 | PyTorch Dataset (Fast .npy Loading)

Loads pre-processed bipolar `.npy` files — no parquet parsing, no cache bloat.
Each `__getitem__` call is a single `np.load()` (~0.5 ms vs ~50 ms for parquet).

**No spectrogram here** — nnAudio computes that inside the model on GPU.

In [ ]:
class HMSDataset(Dataset):
    def __init__(self, df, mode="train", augment=False):
        self.df = df.reset_index(drop=True)
        self.mode = mode
        self.augment = augment
        self.npz_paths = [
            str(PROCESSED_DIR / f"{eid}.npz")
            for eid in self.df["eeg_id"].values
        ]

    def __len__(self):
        return len(self.df)

    def _augment_eeg(self, eeg):
        """Augmentations on normalized EEG signal."""
        noise_std = np.random.uniform(0.005, 0.015)
        eeg = eeg + np.random.normal(0, noise_std, eeg.shape).astype(np.float32)
        scale = np.random.uniform(0.8, 1.2)
        eeg = eeg * scale
        if random.random() < 0.3:
            n_drop = random.randint(1, 2)
            drop_idx = random.sample(range(16), n_drop)
            eeg[drop_idx] = 0.0
        return eeg

    def _get_label(self, idx):
        """
        Get label for sample. Priority:
        1. Soft labels from expert votes (real probability distribution)
        2. One-hot with label smoothing (fallback)
        """
        row = self.df.iloc[idx]

        # Soft labels from vote counts
        if CFG.use_soft_labels and row.get("soft_labels") is not None:
            label = np.array(row["soft_labels"], dtype=np.float32)
            return torch.tensor(label, dtype=torch.float32)

        # Fallback: one-hot with label smoothing
        label = torch.zeros(CFG.num_classes, dtype=torch.float32)
        label[int(row["class_label"])] = 1.0

        if CFG.label_smoothing > 0:
            label = label * (1 - CFG.label_smoothing) + CFG.label_smoothing / CFG.num_classes

        return label

    def __getitem__(self, idx):
        data = np.load(self.npz_paths[idx])
        eeg = data["eeg"]        # [16, 10000] normalized
        stats = data["stats"]    # [32] raw channel means + stds

        if self.augment:
            eeg = self._augment_eeg(eeg)

        eeg_tensor = torch.tensor(eeg, dtype=torch.float32)
        stats_tensor = torch.tensor(stats, dtype=torch.float32)

        if self.mode == "test":
            return eeg_tensor, stats_tensor

        label = self._get_label(idx)
        return eeg_tensor, stats_tensor, label

## MixUp Augmentation

The Keras notebook applies MixUp via `keras_cv.layers.MixUp(alpha=2.0)` on every batch.
Here's the PyTorch equivalent, applied during training.

In [ ]:
def mixup(eeg, stats, targets, alpha=2.0):
    """
    MixUp augmentation — blends pairs of samples, stats, and labels.
    Equivalent to keras_cv.layers.MixUp(alpha=2.0).
    """
    indices = torch.randperm(eeg.size(0), device=eeg.device)
    lam = torch.distributions.Beta(alpha, alpha).sample().to(eeg.device)
    mixed_eeg = lam * eeg + (1 - lam) * eeg[indices]
    mixed_stats = lam * stats + (1 - lam) * stats[indices]
    mixed_targets = lam * targets + (1 - lam) * targets[indices]
    return mixed_eeg, mixed_stats, mixed_targets

# 🤖 | Model Architecture

4 components:
1. **nnAudio STFT**: Raw EEG → spectrogram (trainable, on GPU)
2. **EfficientNetV2-B2**: Feature extraction (pretrained ImageNet, same as Keras preset)
3. **Frequency pooling**: Collapse frequency axis, preserve time axis
4. **Bidirectional GRU**: Model temporal evolution → classify

In [ ]:
class SpectrogramModel(nn.Module):
    def __init__(self, cfg=CFG):
        super().__init__()
        self.cfg = cfg

        # ==============================================================
        # Stage 1: nnAudio STFT — raw EEG → spectrogram on GPU
        # ==============================================================
        self.stft = STFT(
            n_fft=cfg.n_fft,
            hop_length=cfg.hop_length,
            sr=cfg.eeg_sample_rate,
            trainable=cfg.trainable_stft,
            output_format="Magnitude",
        )
        self.log_eps = 1e-6

        # ==============================================================
        # Stage 2: CNN backbone
        # ==============================================================
        self.backbone = timm.create_model(
            cfg.backbone,
            pretrained=True,
            in_chans=3,
            features_only=True,
        )
        with torch.no_grad():
            dummy = torch.randn(1, 3, 64, 64)
            backbone_out = self.backbone(dummy)
        backbone_channels = backbone_out[-1].shape[1]
        print(f"Backbone output channels: {backbone_channels}")

        # ==============================================================
        # Stage 3: GRU — reads CNN features across time
        # ==============================================================
        self.gru = nn.GRU(
            input_size=backbone_channels,
            hidden_size=cfg.gru_hidden,
            num_layers=cfg.gru_layers,
            batch_first=True,
            bidirectional=True,
            dropout=cfg.dropout if cfg.gru_layers > 1 else 0.0,
        )

        # ==============================================================
        # Stage 4: Stats side branch
        # ==============================================================
        self.stats_mlp = nn.Sequential(
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(64, 32),
        )

        # ==============================================================
        # Stage 5: Classification head (GRU output + stats features)
        # ==============================================================
        gru_out_dim = cfg.gru_hidden * 2  # bidirectional
        self.head = nn.Sequential(
            nn.Dropout(cfg.dropout),
            nn.Linear(gru_out_dim + 32, 128),
            nn.ReLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(128, cfg.num_classes),
        )

    def make_spectrograms(self, eeg):
        """Raw EEG → 4-chain spectrogram, cropped to 0-30 Hz."""
        batch, channels, time = eeg.shape
        x = eeg.reshape(batch * channels, time)
        x = self.stft(x)
        freq_bins, time_frames = x.shape[1], x.shape[2]
        x = x.reshape(batch, channels, freq_bins, time_frames)
        max_bin = int(30 / (self.cfg.eeg_sample_rate / self.cfg.n_fft)) + 1
        ll = x[:, 0:4, :max_bin, :].mean(dim=1)
        rl = x[:, 4:8, :max_bin, :].mean(dim=1)
        lp = x[:, 8:12, :max_bin, :].mean(dim=1)
        rp = x[:, 12:16, :max_bin, :].mean(dim=1)
        return torch.cat([ll, rl, lp, rp], dim=1)

    def spec_augment(self, spec):
        """
        SpecAugment: mask random frequency and time bands.
        Equivalent to Keras notebook's RandomCutout.
        Applied only during training.
        Input/Output: [batch, freq, time]
        """
        if not self.training:
            return spec

        batch, freq, time = spec.shape

        for _ in range(self.cfg.spec_num_masks):
            # Frequency masking
            f = random.randint(0, self.cfg.spec_freq_mask)
            f0 = random.randint(0, max(freq - f, 1))
            spec[:, f0:f0+f, :] = 0

            # Time masking
            t = random.randint(0, self.cfg.spec_time_mask)
            t0 = random.randint(0, max(time - t, 1))
            spec[:, :, t0:t0+t] = 0

        return spec

    def process_spectrogram(self, spec):
        """Log + normalize + SpecAugment + 3-channel."""
        x = torch.log(spec.clamp(min=self.log_eps))
        mean = x.mean(dim=(1, 2), keepdim=True)
        std = x.std(dim=(1, 2), keepdim=True) + 1e-6
        x = (x - mean) / std

        # SpecAugment after normalization (only during training)
        x = self.spec_augment(x)

        return x.unsqueeze(1).repeat(1, 3, 1, 1)

    def forward(self, eeg, stats):
        """
        Full pipeline: raw EEG + stats → class probabilities.
        Input:  eeg   [batch, 16, 10000]
                stats [batch, 32]
        Output: probs [batch, 6]
        """
        # Main path: EEG → STFT → SpecAugment → CNN → GRU
        spec = self.make_spectrograms(eeg)
        img = self.process_spectrogram(spec)
        features = self.backbone(img)
        fmap = features[-1]
        x = fmap.mean(dim=2)
        x = x.permute(0, 2, 1)
        x = x.float()
        x, _ = self.gru(x)
        x = x[:, -1, :]

        # Side path: stats → MLP
        s = self.stats_mlp(stats.float())

        # Combine and classify
        combined = torch.cat([x, s], dim=1)
        logits = self.head(combined)
        return F.softmax(logits, dim=1)

## Model Sanity Check

Verify shapes at each stage with dummy data.

In [ ]:
# Sanity check on CPU only — no GPU memory used
model_cpu = SpectrogramModel(CFG)  # stays on CPU

dummy_eeg = torch.randn(2, CFG.num_bipolar_channels, CFG.eeg_samples)
dummy_stats = torch.randn(2, 32)
with torch.no_grad():
    spec = model_cpu.make_spectrograms(dummy_eeg)
    print(f"After STFT:          {spec.shape}")
    img = model_cpu.process_spectrogram(spec)
    print(f"After log+norm+3ch:  {img.shape}")
    feats = model_cpu.backbone(img)
    print(f"After CNN backbone:  {feats[-1].shape}")
    pooled = feats[-1].mean(dim=2)
    print(f"After freq pooling:  {pooled.shape}")
    out = model_cpu(dummy_eeg, dummy_stats)
    print(f"Final output:        {out.shape}")
    print(f"Prob sum (expect 1): {out.sum(dim=1).numpy()}")

total_params = sum(p.numel() for p in model_cpu.parameters())
train_params = sum(p.numel() for p in model_cpu.parameters() if p.requires_grad)
model_size_mb = sum(p.numel() * p.element_size() for p in model_cpu.parameters()) / 1e6
print(f"\nTotal params:     {total_params:,}")
print(f"Trainable params: {train_params:,}")
print(f"Model size:       {model_size_mb:.1f} MB")

del model_cpu, dummy_eeg, dummy_stats, spec, img, feats, pooled, out
gc.collect()

# 🔪 | Data Split

**Identical to Keras notebook**: `StratifiedGroupKFold` with 5 folds,
grouped by `patient_id` to prevent leakage, stratified by `class_label`.

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)

df["fold"] = -1
df.reset_index(drop=True, inplace=True)
for fold, (train_idx, valid_idx) in enumerate(
    sgkf.split(df, y=df["class_label"], groups=df["patient_id"])
):
    df.loc[valid_idx, "fold"] = fold

print("Samples per fold and class:")
display(df.groupby(["fold", "class_name"])[["eeg_id"]].count().unstack())

## Build Train & Validation Sets

Helper function to create DataLoaders for any fold.
Called inside the training loop for each fold.

In [ ]:
sample_df = df.groupby("spectrogram_id").head(1).reset_index(drop=True)

def build_fold_loaders(fold, stage="stage1"):
    """Build train/valid DataLoaders for a given fold and training stage."""
    train_data = sample_df[sample_df.fold != fold].reset_index(drop=True)
    valid_data = sample_df[sample_df.fold == fold].reset_index(drop=True)

    # Stage 2: filter training set to high-quality samples only
    if stage == "stage2":
        train_data = train_data[train_data.total_votes >= CFG.min_votes_stage2].reset_index(drop=True)
        print(f"  Stage 2: {len(train_data)} high-quality train samples "
              f"(≥{CFG.min_votes_stage2} votes)")

    train_dataset = HMSDataset(train_data, mode="train", augment=True)
    valid_dataset = HMSDataset(valid_data, mode="valid", augment=False)

    train_loader = DataLoader(
        train_dataset,
        batch_size=CFG.batch_size,
        shuffle=True,
        num_workers=CFG.num_workers,
        pin_memory=True,
        drop_last=True,
        persistent_workers=True if CFG.num_workers > 0 else False,
    )
    valid_loader = DataLoader(
        valid_dataset,
        batch_size=CFG.batch_size,
        shuffle=False,
        num_workers=CFG.num_workers,
        pin_memory=True,
        drop_last=False,
        persistent_workers=True if CFG.num_workers > 0 else False,
    )
    return train_loader, valid_loader, valid_data

# Quick check
train_loader, valid_loader, _ = build_fold_loaders(0, stage="stage1")
print(f"Fold 0 Stage 1: {len(train_loader)} train batches, {len(valid_loader)} valid batches")
train_loader_s2, _, _ = build_fold_loaders(0, stage="stage2")
print(f"Fold 0 Stage 2: {len(train_loader_s2)} train batches (high-quality only)")
del train_loader_s2

## Dataset Sanity Check

In [ ]:
sample_eeg, sample_stats, sample_label = next(iter(train_loader))
print(f"EEG batch:   {sample_eeg.shape}")      # [batch, 16, 10000]
print(f"Stats batch: {sample_stats.shape}")     # [batch, 32]
print(f"Label batch: {sample_label.shape}")     # [batch, 6]
print(f"\nStats sample (first 16 = means, last 16 = stds):")
print(f"  Means range: [{sample_stats[0,:16].min():.2f}, {sample_stats[0,:16].max():.2f}]")
print(f"  Stds range:  [{sample_stats[0,16:].min():.2f}, {sample_stats[0,16:].max():.2f}]")

# Plot raw EEG channels for first sample
fig, axes = plt.subplots(4, 4, figsize=(16, 10))
eeg_np = sample_eeg[0].numpy()
label_name = CFG.label2name[sample_label[0].argmax().item()]

for i in range(16):
    ax = axes[i // 4][i % 4]
    ax.plot(eeg_np[i], linewidth=0.5)
    chain = CHAIN_ORDER[i // 4]
    pair = BIPOLAR_PAIRS[i]
    std_val = sample_stats[0, 16 + i].item()  # show raw std in title
    ax.set_title(f"{chain}: {pair[0]}-{pair[1]} (σ={std_val:.1f})", fontsize=9)
    ax.set_xlim(0, CFG.eeg_samples)
    ax.tick_params(labelsize=7)

fig.suptitle(f"Normalized Bipolar EEG — Label: {label_name}", fontsize=14)
plt.tight_layout()
plt.show()

# 🔍 | Loss Function

KL Divergence — same as `keras.losses.KLDivergence()` in the Keras notebook.

In [ ]:
class KLDivLoss(nn.Module):
    """
    KL Divergence loss.
    PyTorch KLDivLoss expects log-probabilities, so we log() our softmax output.
    """
    def __init__(self):
        super().__init__()
        self.loss_fn = nn.KLDivLoss(reduction="batchmean")

    def forward(self, preds, targets):
        return self.loss_fn(torch.log(preds.clamp(min=1e-8)), targets)

criterion = KLDivLoss()

# ⚓ | Optimizer & LR Schedule

**Differential learning rates** — the Keras notebook uses a single LR for everything.
Here we use separate rates because our model has components at different training maturity:
- STFT: learning from scratch → highest LR
- Backbone: pretrained ImageNet → lowest LR
- GRU + head: new layers → medium LR

The schedule shape matches the Keras notebook: linear warmup → cosine decay.

In [ ]:
def build_optimizer(model, cfg=CFG, stage="stage1"):
    """AdamW with differential learning rates per component."""
    lr_factor = 1.0 if stage == "stage1" else cfg.lr_stage2_factor
    return torch.optim.AdamW([
        {"params": model.stft.parameters(),
         "lr": cfg.lr_stft * lr_factor, "name": "stft"},
        {"params": model.backbone.parameters(),
         "lr": cfg.lr_backbone * lr_factor, "name": "backbone"},
        {"params": list(model.gru.parameters())
                 + list(model.stats_mlp.parameters())
                 + list(model.head.parameters()),
         "lr": cfg.lr_head * lr_factor, "name": "gru_head"},
    ], weight_decay=cfg.weight_decay)


def get_cosine_schedule(optimizer, epochs, warmup_epochs=3):
    """Cosine annealing with linear warmup."""
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return epoch / warmup_epochs
        progress = (epoch - warmup_epochs) / (epochs - warmup_epochs)
        return 0.5 * (1 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# --- Visualize 2-stage LR schedule ---
lr_names = ["stft", "backbone", "gru_head"]
lr_bases = [CFG.lr_stft, CFG.lr_backbone, CFG.lr_head]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Stage 1
for name, base_lr in zip(lr_names, lr_bases):
    lrs = []
    for e in range(CFG.epochs_stage1):
        if e < 3:
            factor = e / 3
        else:
            progress = (e - 3) / (CFG.epochs_stage1 - 3)
            factor = 0.5 * (1 + math.cos(math.pi * progress))
        lrs.append(base_lr * factor)
    ax1.plot(lrs, marker="o", label=name)
ax1.set_title(f"Stage 1: All data ({CFG.epochs_stage1} epochs)")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("LR"); ax1.legend(); ax1.grid(True, alpha=0.3)

# Stage 2
for name, base_lr in zip(lr_names, lr_bases):
    s2_base = base_lr * CFG.lr_stage2_factor
    lrs = []
    warmup = 1  # shorter warmup for stage 2
    for e in range(CFG.epochs_stage2):
        if e < warmup:
            factor = e / warmup
        else:
            progress = (e - warmup) / (CFG.epochs_stage2 - warmup)
            factor = 0.5 * (1 + math.cos(math.pi * progress))
        lrs.append(s2_base * factor)
    ax2.plot(lrs, marker="o", label=name)
ax2.set_title(f"Stage 2: High-quality only ({CFG.epochs_stage2} epochs, {CFG.lr_stage2_factor}x LR)")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("LR"); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 🚂 | Training & Validation Functions

Includes:
- **Mixed precision** (`autocast` + `GradScaler`) — PyTorch equivalent of Keras `mixed_float16`
- **MixUp** on every training batch — matches Keras notebook's always-on augmentation
- **Gradient clipping** — essential for GRU stability

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, epoch,
                    total_epochs=20, cfg=CFG):
    """Train one epoch with MixUp and stats side features."""
    model.train()
    scaler = GradScaler(enabled=cfg.use_amp)
    running_loss = 0.0
    num_batches = 0

    pbar = tqdm(loader, desc=f"Train Epoch {epoch+1}/{total_epochs}")
    for eeg, stats, labels in pbar:
        eeg = eeg.to(device, non_blocking=True)
        stats = stats.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        eeg, stats, labels = mixup(eeg, stats, labels, alpha=2.0)

        with autocast(enabled=cfg.use_amp):
            preds = model(eeg, stats)
            loss = criterion(preds, labels)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg.max_grad_norm)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        num_batches += 1

        gpu_gb = torch.cuda.memory_allocated() / 1e9
        pbar.set_postfix({
            "loss": f"{running_loss / num_batches:.4f}",
            "gpu": f"{gpu_gb:.1f}GB"
        })

    return running_loss / num_batches


@torch.no_grad()
def validate(model, loader, criterion, device, epoch,
             total_epochs=20, cfg=CFG):
    """Validate one epoch."""
    model.eval()
    running_loss = 0.0
    num_batches = 0

    pbar = tqdm(loader, desc=f"Valid Epoch {epoch+1}/{total_epochs}")
    for eeg, stats, labels in pbar:
        eeg = eeg.to(device, non_blocking=True)
        stats = stats.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with autocast(enabled=cfg.use_amp):
            preds = model(eeg, stats)
            loss = criterion(preds, labels)

        running_loss += loss.item()
        num_batches += 1
        pbar.set_postfix({"val_loss": f"{running_loss / num_batches:.4f}"})

    return running_loss / num_batches

# 🏋️ | 5-Fold Cross-Validation + 2-Stage Training

For each fold:
- **Stage 1** (20 epochs): Train on ALL training data with cosine LR schedule
- **Stage 2** (10 epochs): Fine-tune on high-quality samples (≥10 expert votes) with reduced LR

This follows the 1st place solution's approach: the model learns general
patterns first, then specializes on the most reliable annotations.

## Training Loop

The model is created fresh for each fold. Only ONE model lives on GPU at a time.

In [ ]:
def train_fold(fold, cfg=CFG):
    """
    Train one fold with 2-stage training.
    Returns: best_val_loss, history dict
    """
    print(f"\n{'='*60}")
    print(f"FOLD {fold}")
    print(f"{'='*60}")

    # Clean GPU
    gc.collect()
    torch.cuda.empty_cache()
    set_seed(cfg.seed + fold)  # different seed per fold for diversity

    # Create model
    model = SpectrogramModel(cfg).to(cfg.device)
    print(f"GPU after model: {torch.cuda.memory_allocated()/1e9:.2f} GB")

    checkpoint_path = str(MODELS_DIR / f"best_model_fold{fold}.pt")
    history = {"train_loss": [], "val_loss": [], "lr": [], "stage": []}
    best_val_loss = float("inf")
    best_epoch = -1

    # ==============================================================
    # STAGE 1: Train on ALL data
    # ==============================================================
    print(f"\n--- Stage 1: All data, {cfg.epochs_stage1} epochs ---")
    train_loader, valid_loader, _ = build_fold_loaders(fold, stage="stage1")
    optimizer = build_optimizer(model, cfg, stage="stage1")
    scheduler = get_cosine_schedule(optimizer, cfg.epochs_stage1, warmup_epochs=3)

    for epoch in range(cfg.epochs_stage1):
        current_lrs = {pg["name"]: f"{pg['lr']:.2e}" for pg in optimizer.param_groups}
        print(f"\n[S1] Epoch {epoch+1}/{cfg.epochs_stage1} | LRs: {current_lrs}")

        train_loss = train_one_epoch(
            model, train_loader, optimizer, criterion, cfg.device, epoch,
            total_epochs=cfg.epochs_stage1)
        val_loss = validate(
            model, valid_loader, criterion, cfg.device, epoch,
            total_epochs=cfg.epochs_stage1)
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["lr"].append(optimizer.param_groups[1]["lr"])
        history["stage"].append(1)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            torch.save({
                "epoch": epoch, "stage": 1,
                "model_state_dict": model.state_dict(),
                "val_loss": val_loss,
            }, checkpoint_path)
            print(f"  ✓ Best model saved (val_loss: {val_loss:.4f})")
        else:
            print(f"  val_loss: {val_loss:.4f} (best: {best_val_loss:.4f} @ S1 epoch {best_epoch+1})")

    print(f"\nStage 1 done. Best val_loss: {best_val_loss:.4f}")

    # ==============================================================
    # STAGE 2: Fine-tune on HIGH-QUALITY data only
    # ==============================================================
    print(f"\n--- Stage 2: High-quality data (≥{cfg.min_votes_stage2} votes), "
          f"{cfg.epochs_stage2} epochs, {cfg.lr_stage2_factor}x LR ---")

    # Load best stage 1 checkpoint before fine-tuning
    ckpt = torch.load(checkpoint_path, map_location=cfg.device)
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"  Loaded best Stage 1 model (val_loss: {ckpt['val_loss']:.4f})")

    train_loader_s2, valid_loader, _ = build_fold_loaders(fold, stage="stage2")
    optimizer_s2 = build_optimizer(model, cfg, stage="stage2")
    scheduler_s2 = get_cosine_schedule(optimizer_s2, cfg.epochs_stage2, warmup_epochs=1)

    for epoch in range(cfg.epochs_stage2):
        current_lrs = {pg["name"]: f"{pg['lr']:.2e}" for pg in optimizer_s2.param_groups}
        print(f"\n[S2] Epoch {epoch+1}/{cfg.epochs_stage2} | LRs: {current_lrs}")

        train_loss = train_one_epoch(
            model, train_loader_s2, optimizer_s2, criterion, cfg.device, epoch,
            total_epochs=cfg.epochs_stage2)
        val_loss = validate(
            model, valid_loader, criterion, cfg.device, epoch,
            total_epochs=cfg.epochs_stage2)
        scheduler_s2.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["lr"].append(optimizer_s2.param_groups[1]["lr"])
        history["stage"].append(2)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = cfg.epochs_stage1 + epoch
            torch.save({
                "epoch": best_epoch, "stage": 2,
                "model_state_dict": model.state_dict(),
                "val_loss": val_loss,
            }, checkpoint_path)
            print(f"  ✓ Best model saved (val_loss: {val_loss:.4f})")
        else:
            print(f"  val_loss: {val_loss:.4f} (best: {best_val_loss:.4f})")

    # Cleanup
    del model, optimizer, optimizer_s2, scheduler, scheduler_s2
    del train_loader, valid_loader, train_loader_s2
    gc.collect()
    torch.cuda.empty_cache()

    print(f"\nFold {fold} done. Best val_loss: {best_val_loss:.4f}")
    return best_val_loss, history


# ==============================================================
# Run all folds
# ==============================================================
fold_results = {}
all_histories = {}

for fold in CFG.run_folds:
    try:
        best_val_loss, history = train_fold(fold)
        fold_results[fold] = best_val_loss
        all_histories[fold] = history
    except RuntimeError as e:
        print(f"\n✗ Fold {fold} failed: {e}")
        if "out of memory" in str(e).lower():
            print("  → Restart kernel, reduce batch_size")
        gc.collect()
        torch.cuda.empty_cache()
        break  # don't try more folds if OOM

# ==============================================================
# Summary
# ==============================================================
print("\n" + "=" * 60)
print("CROSS-VALIDATION RESULTS")
print("=" * 60)
for fold, val_loss in fold_results.items():
    print(f"  Fold {fold}: val_loss = {val_loss:.4f}")
if fold_results:
    cv_score = np.mean(list(fold_results.values()))
    cv_std = np.std(list(fold_results.values()))
    print(f"\n  CV = {cv_score:.4f} ± {cv_std:.4f}")

## Training History (All Folds)

In [ ]:
n_folds_done = len(all_histories)
fig, axes = plt.subplots(1, n_folds_done, figsize=(6*n_folds_done, 5))
if n_folds_done == 1:
    axes = [axes]

for ax, (fold, hist) in zip(axes, all_histories.items()):
    epochs_range = range(1, len(hist["train_loss"]) + 1)
    ax.plot(epochs_range, hist["train_loss"], "b-o", markersize=3, label="train")
    ax.plot(epochs_range, hist["val_loss"], "r-o", markersize=3, label="valid")

    # Mark stage boundary
    s1_epochs = sum(1 for s in hist["stage"] if s == 1)
    ax.axvline(x=s1_epochs + 0.5, color="gray", linestyle="--", alpha=0.5)
    ax.text(s1_epochs * 0.5, ax.get_ylim()[1] * 0.95, "Stage 1",
            ha="center", fontsize=9, color="gray")
    ax.text(s1_epochs + (len(hist['stage']) - s1_epochs) * 0.5, ax.get_ylim()[1] * 0.95,
            "Stage 2", ha="center", fontsize=9, color="gray")

    ax.set_title(f"Fold {fold} (best: {fold_results[fold]:.4f})")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("KL Divergence Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle(f"CV = {cv_score:.4f} ± {cv_std:.4f}", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# 🧪 | Prediction (Ensemble of All Folds)

Load the best checkpoint from each fold, predict on test set,
then average the probabilities. This is the standard Kaggle
submission strategy for cross-validated models.

In [ ]:
# Build test DataLoader
test_dataset = HMSDataset(test_df, mode="test", augment=False)
test_loader = DataLoader(
    test_dataset,
    batch_size=min(CFG.batch_size, len(test_df)),
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=True,
    drop_last=False,
)
print(f"Test samples: {len(test_dataset)} | Test batches: {len(test_loader)}")

## Run Inference Across All Folds

In [ ]:
@torch.no_grad()
def predict(model, loader, device, use_amp=CFG.use_amp):
    """Run inference. Returns numpy predictions."""
    model.eval()
    all_preds = []
    for eeg, stats in tqdm(loader, desc="Predicting"):
        eeg = eeg.to(device, non_blocking=True)
        stats = stats.to(device, non_blocking=True)
        with autocast(enabled=use_amp):
            preds = model(eeg, stats)
        all_preds.append(preds.cpu().numpy())
    return np.concatenate(all_preds, axis=0)


# Predict with each fold's best model and average
all_fold_preds = []

for fold in CFG.run_folds:
    ckpt_path = str(MODELS_DIR / f"best_model_fold{fold}.pt")
    if not Path(ckpt_path).exists():
        print(f"  ⚠ Fold {fold} checkpoint not found, skipping")
        continue

    print(f"\nFold {fold}:")
    gc.collect()
    torch.cuda.empty_cache()

    model = SpectrogramModel(CFG).to(CFG.device)
    ckpt = torch.load(ckpt_path, map_location=CFG.device)
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"  Loaded: stage={ckpt.get('stage','?')}, val_loss={ckpt['val_loss']:.4f}")

    fold_preds = predict(model, test_loader, CFG.device)
    all_fold_preds.append(fold_preds)
    np.save(str(RESULTS_DIR / f"pytorch_preds_fold{fold}.npy"), fold_preds)
    print(f"  ✓ Predictions saved ({fold_preds.shape})")

    del model
    gc.collect()
    torch.cuda.empty_cache()

# Average across folds
preds = np.mean(all_fold_preds, axis=0)
print(f"\n✓ Ensemble of {len(all_fold_preds)} folds")
print(f"  Predictions shape: {preds.shape}")
print(f"  Prob sum check: {preds[0].sum():.4f}")

# 📩 | Submission

In [ ]:
target_cols = [x.lower() + "_vote" for x in CFG.class_names]
pred_df = test_df[["eeg_id"]].copy()
pred_df[target_cols] = preds.tolist()

# Merge with sample submission
sub_df = pd.read_csv(BASE_PATH / "sample_submission.csv")
sub_df = sub_df[["eeg_id"]].copy()
sub_df = sub_df.merge(pred_df, on="eeg_id", how="left")

submission_path = str(RESULTS_DIR / "submission.csv")
sub_df.to_csv(submission_path, index=False)
print(f"✓ Submission saved to: {submission_path}")
print(f"  Ensemble of {len(all_fold_preds)} fold predictions")
print(f"\nSubmission preview:")
print(sub_df.head())

# 🔀 | Ensembling with Keras Predictions (Optional)

If you ran the Keras notebook, you can blend PyTorch + Keras predictions
for an even stronger submission.

In [ ]:
# Uncomment to ensemble with Keras

# keras_preds = np.load(str(RESULTS_DIR / "keras_preds.npy"))
# pytorch_preds = preds  # already averaged across folds
# blend = 0.5 * keras_preds + 0.5 * pytorch_preds
# blend_df = test_df[["eeg_id"]].copy()
# blend_df[target_cols] = blend.tolist()
# blend_df.to_csv(str(RESULTS_DIR / "submission_blend.csv"), index=False)
# print("✓ Blended submission saved")

# 🔀 | Ensembling with Keras Predictions (Optional)

If you ran the Keras notebook, you can ensemble predictions from both.
Just save Keras predictions by adding this to the Keras notebook after `model.predict()`:
```python
np.save('results/keras_preds_fold0.npy', preds)
```

In [ ]:
# Uncomment to ensemble

# keras_path = RESULTS_DIR / "keras_preds_fold0.npy"
# pytorch_path = RESULTS_DIR / f"pytorch_preds_fold{CFG.fold}.npy"
#
# if keras_path.exists():
#     keras_preds = np.load(str(keras_path))
#     pytorch_preds = np.load(str(pytorch_path))
#
#     # Adjust weights based on val_loss (lower loss → higher weight)
#     w_keras, w_pytorch = 0.3, 0.7
#     ensemble = w_keras * keras_preds + w_pytorch * pytorch_preds
#
#     ens_df = sub_df[["eeg_id"]].copy()
#     ens_df[target_cols] = ensemble.tolist()
#     ens_df.to_csv(str(RESULTS_DIR / "submission_ensemble.csv"), index=False)
#     print("✓ Ensemble submission saved")
# else:
#     print(f"Keras predictions not found at {keras_path}")
#     print("Add np.save('results/keras_preds_fold0.npy', preds) to Keras notebook")

# 📊 | Inspect Learned STFT (Optional)

One advantage of trainable STFT: we can see what the model learned.

In [ ]:
if CFG.trainable_stft:
    try:
        # Visualize learned vs initial STFT kernels
        learned = model.stft.wsin.detach().cpu().numpy()
        print(f"STFT kernel shape: {learned.shape}")

        fig, axes = plt.subplots(2, 4, figsize=(16, 6))
        for i, ax in enumerate(axes.flat):
            if i < min(8, learned.shape[0]):
                ax.plot(learned[i, 0, :], alpha=0.8)
                ax.set_title(f"Filter {i}", fontsize=9)
                ax.tick_params(labelsize=7)
        plt.suptitle("Learned STFT Filters", fontsize=14)
        plt.tight_layout()
        plt.show()
    except AttributeError as e:
        print(f"Could not extract STFT params: {e}")
        print("nnAudio version may use different attribute names.")

# 📌 | Reference
* [HMS-HBAC: ResNet34d Baseline [Training]](https://www.kaggle.com/code/ttahara/hms-hbac-resnet34d-baseline-training)
* [EfficientNetB2 Starter - [LB 0.57]](https://www.kaggle.com/code/cdeotte/efficientnetb2-starter-lb-0-57)
* [nnAudio Documentation](https://kinwaicheuk.github.io/nnAudio/)
* [timm Model Zoo](https://huggingface.co/timm)